# Third-medium Contact

---
authors:
  - Nguyen Anh Tuan
  - Julian Norato

---
*School of Mechanical, Aerospace, and Manufacturing Engineering, University of Connecticut, 191 Auditorium Road, U3139, CT, 06269, USA*

The topic solve the C-shape models with the Third-medium Contact (TMC) approach, which is obtained by adding a regularization term to the base material strain energy density.

In this topic, we focus on
 - How to set up and solve large deformation contact problem with the TMC method.
 - The effect of different regularizaiton terms on convergence behavior and the third medium.
 - Adaptive load-steping strategy for highly nonlinear problems.
 - How to extract the nodal reaction force in a nonlinear, predescribed displacement control.

## Geometry

The C-shape benchmark problem, first introduced by [[1](https://doi.org/10.48550/arXiv.2010.14277)], which consists of C-shaped body of length $L$, height $H = 0.5 L$  and thickness $T = 0.1L$, clampled on the left side and loaded by a vertical prescribed displacement at the right-top corner node of the upper beam. The third-medium contact fills the gap between two beams and is further extended by an additional column of elements from $x = L$ to $x = L + \Delta L$ to completely embed the potential contact region. Firstly, the model is discretized with quadrilateral elements for the earlier regularizaiton technique, then the linear triangle and tetrahedral are applied for the latest ones


In [1]:
# --------------------------------------------------------------------
# Import packages
#---------------------------------------------------------------------

using Gridap
using Gridap.Fields
using Gridap.Geometry
using Gridap.CellData
using LineSearches: BackTracking
using Statistics

#-----------------------------------------------------------------------
# Geometry - C_shape body
#-----------------------------------------------------------------------
L, H = 1.24, 0.6                    #length, and height
h_bot = 0.2*H                       # yellow bottom thickness
h_mid = 0.6*H                       # brown middle thickness
h_top = 0.2*H                       # yellow top thickness
d = L/30;


#-----------------------------------------------------------------------
# Discretisation
#-----------------------------------------------------------------------
Nx, Ny = 31, 15                     # cells along x and y within C-shape
domain = (0, L, 0, H)
partition = (Nx,Ny)
model = CartesianDiscreteModel(domain, partition)
Ω = Triangulation(model)

# Tag cells by centroid coordinate - mark cells
cell_points = get_cell_coordinates(Ω)          # 4 corner points per cell
cell_centroids = lazy_map(mean, cell_points)   # ← one centroid per cell
centroid_y = get_array(lazy_map(p -> p[2], cell_centroids))
# Material IDs (1 = solid, 2 = third medium)
material_id = zeros(Int, num_cells(model))
for (i, q) in enumerate(cell_centroids)
    (x,y) = q 
    if ((y ≤ h_bot || y ≥ (h_bot + h_mid) || x ≤ 0.2*H) && x ≤ 1.2)    # yellow layers
        material_id[i] = 1
    else                                                 # brown middle
        material_id[i] = 2
    end
end
material_field = CellField(material_id, Ω);   # for VTK

## Third-medium contact formulation

To enable robust simulation of large deformation self-contact while accommodating severe distortion, a fictitious hyperelastic continuum is introduced to represent the void region between the contacting surfaces. In this methodology, the third medium $\Omega_m$ acts as a layer that undergoes high compression in the pre-contact phase while while rapidly increase the stiffness as compressed to near-zero volume, allowing the transfer of contact tractions. In this way, the contact problem reduces to standard hyperelastic problem, thus avoiding the explicit contact search and the imposition of inequality constraint as classical contact algorithm.

Under the assumption of finite deformations, a material partical $\mathbf{X}$ in the undeformed (or reference) configuration is mapped onto $\mathbf{x} = \boldsymbol{\Phi}(\mathbf{X})$ in the deformed (or current) configuration. The deformation gradient is given by $\mathbf{F}(\mathbf{X}) = \nabla \boldsymbol{\Phi}(\mathbf{X})$. The relationship between the volume of infinitesimal element in the current configuration and the corresponding element in the reference configuration is given by $\mathrm{d}\Omega = J(\mathbf{X}) \mathrm{d}\Omega_0$, where $J(\mathbf{X}) = |\mathbf{F}(\mathbf{X})| = det(\mathbf{F}(\mathbf{X}))>0$ is the Jacobian of the deformation gradient. The strain is defined in terms of the Green Lagrange strain tensor $\mathbf{E} = \frac{1}{2} \left( \mathbf{C} - \mathbf{I}\right)$, where $\mathbf{C} = \mathbf{F}^{\top} \mathbf{F}$ is the Cauchy deformation tensor. Generally, the body and third-medium are modeled with the same compresible Neo-Hookean strain energy density $\Psi(\mathbf{C})$ in the form:

$$\Psi := \frac{\mu}{2}\left( \operatorname{tr} \boldsymbol{C} - 3 - 3 \ln J \right) + \frac{\lambda}{2}\left(\ln J\right)^{2}$$

where $K$ and $\mu$ denote the bulk and shear moduli, respectively. To keep the influence of the third-medium negligible before contact, its strain energy density is scaled by a scaling parameter $\gamma$. Thereby avoiding spurious parasitic forces.

$$\Psi_{TMC} := \gamma \left[\frac{\mu}{2}\left( \operatorname{tr} \boldsymbol{C} - 3 - 3 \ln J \right) + \frac{\lambda}{2}\left(\ln J\right)^{2}\right]$$

When large-deformation occur in the pre-contact phase, the third medium elements become severely distorted, preventing convergence of the nonlinear solver of Newton Raphson algorithm. To mitigate this issue a regularization term $\Psi_r$ is added to the strain energy density that should provide sufficient stabilization while minimally influencing its deformation. The material properties for the body are set from bulk's modulus $K = 5/3$ and $\mu = 5/14$ MPa. By choosing the sufficiently small scaling factor $\gamma = 10^{-6}$, the contribution of the regularization term to the strain energy density within solid regions is made negligible.

In [2]:
#-----------------------------------------------------------------------
# Material parameter
#-----------------------------------------------------------------------

# E = 1.0                 # Young's modulus
# ν = 0.4                 # Poisson ratio
# μ = E / (2 * (1 + ν));                   # Shear modulus 
# K = (E * (1 + ν)) / (3 * (1 - 2ν));      # Bulk modulus
# λ = (E * ν) / ((1 + ν) * (1 - 2ν)); 
# Yellow (stiff)   → K₁, μ₁
γ  = 1.0e-5              # scaling parameter
K₁ = 5.0/3.0;  μ₁ = 5/14; λ₁ = 30/21;

# Brown  (soft)    → K₂, μ₂
K₂ = K₁*γ;  μ₂ = μ₁*γ; λ₂ = λ₁*γ; 

K = 5.0/3.0;  μ = 5/14; λ = 30/21;
# Regularisation parameters


#-----------------------------------------------------------------------
# Constitutive law - Neo-Hookean (per cell)
#-----------------------------------------------------------------------
F(∇u) = one(∇u) + ∇u                   # deformation gradient
C(∇u) = F(∇u)' ⋅ F(∇u)                   #  Cauchy doformation tensor
J(∇u) = sqrt(det(C(∇u)))                # Jacobian of deformation

# Define the Green-Lagrange strain field for the converged solution
E_cauchy(∇u) = 0.5 *(C(∇u) - one(C(∇u))) # Green-Lagrange Strain: E = 0.5(C - I)

# Strain Energy Density (SED) for Neo-Hookean model
function Ψ₀(∇u)
    vol_term = 0.5*K*(log(J(∇u)))^2
    iso_term = 0.5 * μ * ((J(∇u)^(-2.0/3.0))*tr(C(∇u)) - 3.0)    
    return vol_term + iso_term
end

#-----------------------------------------------------------------------
# Find node by coordinate (tolerance = 10e-6)
#-----------------------------------------------------------------------

tol = 1e-6
tip_point = Point(1.2, 0.6)                 # (x,y) coordinate of node

# Get all node coordinates
node_coords = Gridap.Geometry.get_node_coordinates(model) 

# Find node index
let tip_node_id = nothing
for (i, p) in enumerate(node_coords)
    if norm(p - tip_point) < tol
        tip_node_id = i
        break
    end
end

@assert tip_node_id !== nothing "Tip node not found – check coordinates / tolerance"
    # ---- expose only the id to the outer scope ----
    global tip_node_id_global = tip_node_id
end

println("Prescribed displacement at node $tip_node_id_global: (1.2, 0.5)")

#-----------------------------------------------------------------------
# Boundary condition tags
#-----------------------------------------------------------------------

labels = get_face_labeling(model)
entity = num_entities(labels)+1
labels.d_to_dface_to_entity[1][tip_node_id_global] = entity
add_tag!(labels,"Loads",[entity])
add_tag_from_tags!(labels, "Fixed", [1, 3, 7])   # left edge + corners

# Element type
order = 2
degree = 2*order+2
dΩ = Measure(Ω, degree)

# FE space

reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
V = TestFESpace(model, reffe, conformity=:H1,
                dirichlet_tags=["Fixed", "Loads"])

Prescribed displacement at node 511: (1.2, 0.5)


UnconstrainedFESpace()

## Solution strategy

Solving a contact problem with the TMC method requires to find a stationary point of the total potential energy of the system $\Pi(\mathbf{u})$. Several forms of $\Psi_r$ have been proposed, mainly differing in the deformation modes that they penalized: some are formulated inthe displacement alone, others introduced auxiliary fields. It is therefore convenient to collect all unknowns of a given formulation in a single subscribe state $m$. The total potential energy then splits into the elastic energy of body, and third medium with a regularization contribution
$$ \Pi(\mathbf{m}) = \Pi_{el}(\mathbf{u}) + \Pi_r(\mathbf{m}) $$

where the common part collects the elastic energy of the body and of the third medium

$$ \Pi_{el}(\boldsymbol{u}) = \int_{\Omega_{b}} \Psi_{body}(\boldsymbol{u}) \,\mathrm{d} \Omega +
\int_{\Omega_{tm}} \Psi_{tm}(\boldsymbol{u}) \,\mathrm{d} \Omega, $$

with $\Omega_{b}$ and $\Omega_{tm}$ the body and third medium subdomains, and the regularization
part is:

$$ \Pi_{r}(\mathbf{m}) = \int_{\Omega_{tm}} \Psi_{r}(\mathbf{m}) \,\mathrm{d} \Omega, $$

whose specific form, together with the auxiliary fields it involves the load is applied through prescribed displacement only, so no external work term appears. The nonlinear system is solved using Newton solver with line search.

The finite element formulation of hyperelastic materials is well established and will not be further detailed here. To incorporate the regularization of the third medium, the weak form is derived by taking the Gâteaux derivative of $\Pi(\mathbf{m})$ with respect to a variation in the displacement field and auxiliary field.

$$ 
\mathcal{R}(\mathbf{m}; \delta \mathbf{m}) := \delta \Pi(\mathbf{m}; \delta \mathbf{m}) = \delta \Pi_{el}(\mathbf{u}; \delta \mathbf{u}) + \delta \Pi_r(\mathbf{m}; \delta \mathbf{m}),
 $$
where the part collects the elastic energy of the body and the third medium

$$ 
\delta \Pi_{el}(\mathbf{u}; \delta \mathbf{u}) = \int_{\Omega_b} \delta \Psi_{body}(\mathbf{u}; \delta \mathbf{u}) \,\mathrm{d} \Omega + \int_{\Omega_{tm}} \delta \Psi_{tm}(\mathbf{u}; \delta \mathbf{u})\,\mathrm{d} \Omega
$$

In continuum mechanics and finite element method (FEM) for large-deformation elastic, the weak form (based on the principle of virtual work) can be written using the **second Piola-Kirchoff stress tensor** $\mathbf{S} = 2 \frac{\partial \Psi}{\partial \mathbf{C}} = \lambda \, \ln(J)\,\mathbf{C}^{-1} + \mu \, \left(\mathbf{I} - \mathbf{C}^{-1}\right)$. Therefor, the $\delta \Psi_{body}$ and $\delta \Psi_{tm}$ could be obtained as

$$
\delta \Psi_{body} = \mathbf{S} \, \delta \mathbf{E} \\
\delta \Psi_{tm} = \gamma \, \mathbf{S} \, \delta \mathbf{E}
$$

where the variation of the Green-Lagrange strain tensor $\delta \mathbf{E}$ can be expressed in terms of the deformation gradient $\mathbf{F}$ and its variation $\delta \mathbf{F}$ as follow
$$
\delta \mathbf{E} = \frac{1}{2} \left( \delta \mathbf{F}^{\top} \mathbf{F} + \mathbf{F}^{\top} \delta \mathbf{F}  \right) 
$$
Depending on wether you prefer using the displacement $\mathbf{u}$, it could be written in terms of varation of displacement $\mathbf{v} = \delta \mathbf{u}$

$$
\delta \mathbf{E} = \frac{1}{2} \left( \nabla_0(\mathbf{v}) ^{\top} \mathbf{F} + \mathbf{F}^{\top} \nabla_0 (\mathbf{v})  \right) 
$$

where $\nabla_0$ is the gradient with respect to the undeformed configuration
 
whose specific form, together with the auxiliary fields it involves, in detailed in each of the following section.

$$
\delta \Pi_r(\mathbf{m}) = \int_{\Omega} \Psi_r(\mathbf{m}) \,\mathrm{d} \Omega
$$

In [3]:
# Virtual Green-Lagrange strain increment
δE(∇v, ∇u) = 0.5 * (∇v' ⋅ F(∇u) + F(∇u)' ⋅ ∇v)

# Second Piola-Kirchhoff stress
function S(∇u, tag)
    λ, μ = (tag == 1) ? (λ₁, μ₁) : (λ₂, μ₂)
    Cinv = inv(C(∇u))
    return μ*(one(∇u) - Cinv) + λ*log(J(∇u))*Cinv
end

S (generic function with 1 method)

## HuHu regularization

The first regularization was proposed by [[1](https://doi.org/10.48550/arXiv.2010.14277)] used to penalize second-order deformation modes through the Hessian of displacement field $ \mathbb{H} \mathbf{u} $, hence the name "HuHu regularization":

$$
\Psi_r^{Hu}(\mathbf{u}) = \frac{k_r}{2} \mathbb{H} \mathbf{u} \, \mathbb{H} \mathbf{u},
$$

where $k_r = \alpha L^2 K$, with $L$ is a characteristic length of the problem and $\alpha$ is a dimensionless parameter to be chosen as small as possible while still stabilizing the third medium. A value of $\alpha = 10^{-6}$ is adopted in the following

In [4]:
# regularization parameter
const α = 1.0e-6
kᵣ = α*(L^2)*K

# Hessian operation
function ℌ(u)
    return ∇∇(u)
end

#----------------------------------------------------
# Weak form (based on the principle of virtual work)
#----------------------------------------------------
reg_mask = CellField(material_id .== 2, Ω)
function ℝ(u,v)
    based_material = ∫((δE ∘ (∇(v), ∇(u))) ⊙ (S ∘ (∇(u), material_field)))dΩ 
    HuHu_regularization =∫(0.5*kᵣ*(ℌ(u)⊙ℌ(v))* reg_mask)dΩ
    return based_material + HuHu_regularization
end

ℝ (generic function with 1 method)

The load is applied through prescribed displacement, so no external work term appears. The nonlinear system is solved using Newton solver with line search, coupled with an adaptive load stepping strategy to improve robustness and convergence: on failure the state is reset to last converged one and the increment is halved, until the target displacement $\bar{\mathbf{u}}$ is reached or increment falls below $\Delta l_{min}$

In [5]:
#----------------------------------------------------
# Solver - Newton solver with line search
#----------------------------------------------------

nls = NLSolver(show_trace=true, method=:newton,
    linesearch= BackTracking(), iterations = 200)

solver = FESolver(nls)

# Setup adaptive loading strategy or constant loading step

function run_step(x0, load_y, step, cache, fesolver, regularization_term)
    g_clamped = VectorValue(0.0, 0.0)
    g_loaded  = VectorValue(0.0, load_y)        # Vertical prescribed displacement

    U = TrialFESpace(V, [g_clamped, g_loaded])

    op = FEOperator(ℝ, U, V)
    uh = FEFunction(U, x0)
    # Safe solver execution tracking
    local solver_failed = false    

    try
        uh, cache = solve!(uh, fesolver, op, cache)
    catch
        solver_failed = true
    end

    r_vector = Gridap.FESpaces.residual(op, uh)
    r_norm = norm(r_vector)
    # Define your own success criteria
    # Usually, a residual < 1e-6 is excellent for non-linear mechanics
    if r_norm > 1e-6 || solver_failed
        println("!!! Step $step DIVERGED")
        return nothing, nothing, false
    end
    
    # ----- VTK output (includes material_id) -----
    data = ["u" => uh, "material_id"=> material_field]
    mkpath(regularization_term)
    writevtk(
        Ω,"$regularization_term/step_$(lpad(step,3,'0'))";
        cellfields = data
    )    
    return get_free_dof_values(uh), cache, true
end

function run_adaptive_loading(fesolver, regularization_term, target_displacement, adaptive_load, initial_increment, min_increment, max_increment)
    max_load = target_displacement
    last_load = 0.0
    dload = initial_increment
    min_dload = min_increment
    max_dload = max_increment
    load = last_load + dload
    num_iteration = 0
    load_step = 1
    U_int = TrialFESpace(V)
    x_converged = zeros(Float64, num_free_dofs(V))
    cache = nothing

    while abs(load) < abs(max_load)
        increment = sign(max_load)*min(dload, abs(max_load - load))
        trial_load = load + increment

        println("\n>>> LOAD STEP: u_y $trial_load (dload: $increment)")
        
        u_new, new_cache, success = run_step(x_converged, trial_load, load_step, cache, fesolver,regularization_term)
        if success
            # Success: Move state forward
            load_step += 1
            x_converged = u_new
            cache = new_cache
            load = trial_load
            if abs(load) < abs(max_load)
                dload = (dload >= max_dload) ? dload : min(dload * 1.1, abs(max_load - load))
            end          
        else
            # Failure: Backtrack and cut load step
            println("!!! DIVERGENCE: Cutting load step size from $dload to $(dload/2)")
            dload /= 2
            if dload < min_dload
                error("Simulation aborted: Step size dropped below threshold.")
            end
        end    
    end
    return
end

run_adaptive_loading (generic function with 1 method)

A Gauss-Lobatto quadrature rule is employed for third-medium elements to mitigate element inversion, increasing robustness under severe skew deformations.

In [6]:
#run_adaptive_loading(solver, "HuHu_regularization_out", -0.6, true, 0.004, 0.002, 0.005)

## HuHu-LuLu regularization

To reduce the penalization on bending and quadratic compression models, [[2](https://doi.org/https://doi.org/10.1016/j.cma.2024.117595 )] proposed to subtract a term in the Laplacian of the displacement field $\mathbb{L}\mathbf{u}$ from HuHu regularization which leading to the so-called "HuHu-LuLu regularization". :

$$
\Psi_r^{HuLu}(\mathbf{u}) = \frac{k_r}{2} (\mathbb{H} \mathbf{u} \cdot  \mathbb{H} \mathbf{u} - \frac{1}{\operatorname{tr}} \mathbb{L}\mathbf{u} \cdot \mathbb{L}\mathbf{u} ),
$$

In [7]:
# Laplacian operation
function 𝕃(u) #Δ(u) = laplacian(u) = div(grad(u) = tr(∇∇(a))
    return Δ(u)
end

#----------------------------------------------------
# Weak form (based on the principle of virtual work)
#----------------------------------------------------
function ℝ(u,v)
    based_material = ∫((δE ∘ (∇(v), ∇(u))) ⊙ (S ∘ (∇(u), material_field)))dΩ 
    HuLu_regularization =∫(0.5*kᵣ*(ℌ(u)⊙ℌ(v) - (1/3)*𝕃(u)⊙𝕃(u))* reg_mask)dΩ
    return based_material + HuLu_regularization
end

#run_adaptive_loading(solver, "HuHu_LuLu_regularization_out", -0.6, true, 0.002, 0.002, 0.003)

ℝ (generic function with 1 method)

## Peter Wriggers regularizaion

The earlier proposed regularization technique typically requires the use of second-order finite elements, as otherwise the gradient of deformation gradient field $\nabla_0 \mathbf{F}$ vanishes within the element interior. While second-order quadrilateral and hexahedral elements circumvent this, they require more analysis degree of freedom and thus lead to higher computational cost. Furthermore, the gradients of deformation gradient in second-order triangular and tetrahedral elements remain constant within each element, which leads to less effective penalization of the deformation in the third medium.

To circumvent these limitation, [[3](https://doi.org/10.1007/s00466-025-02628-y)] introduced an approximate computation of the regularization terms that stabilizes first-order elements with linear shape functions using an aulixiary field, as it also can be used with linear triangular and tetrahedral elements.

In the two-dimensional case, the underlying idea is to consider curvature at the element level as the primary problem of an unregularized medium, whereas stretch and volume-changed gradient are desired behaviors during the compliant pre-contact phase. Penalizing curvature along, expressed as the gradient of the rotation tensor $\mathbf{R}$ from the polar decomposition $\mathbf{F} = \mathbf{R} \mathbf{U}$, thus yield more compliant medium than the Hessian or Hessian-Laplacian forms above. the rotation tensor can be expressed in terms of a single angle $\varphi$, and the symmetry of $\mathbf{U}$ yields the explicit relation $\tan\varphi = \frac{F_{12} - F_{21}}{F_{11} + F_{22}}$, so that the penalizing $\nabla_0 \mathbf{R}$ reduces to penalizing $\nabla_0 \varphi$. Penalizing the tangent instead of the angle itself expresses the regularization directly through the components of $\mathbf{F}$, avoiding the arctangent, leads to:

$$
\Psi_r^{\tan}(\boldsymbol{u}) = \frac{\gamma}{2}\alpha_r\left(\left\|\nabla\left[\frac{F_{12} -
F_{21}}{F_{11} + F_{22}}\right]\right\|^2 + \|\nabla J\|^2\right).
$$

As only rotation are penalized, stretch deformations remain unconstrained and elements may approach a locally vanishing volume; the second term, involving $\nabla J$, restores control over the volumetric modes. This form still involves second derivatives of $\mathbf{u}$. They are eliminated by introducing two auxiliary scalar fields $p$ and $q$ which approximate $\tan \varphi$ and $J$ in a penalty-like fashion, so that the gradient penalization acts on the auxiliary fields alone:

$$ \Psi_r(\boldsymbol{u}, p, q) = \frac{\gamma}{2}\left[\beta_1\left(\frac{F_{12} - F_{21}}{F_{11}
+ F_{22}} - p\right)^2 + \alpha_r\|\nabla p\|^2\right] + \frac{\gamma}{2}\left[\beta_2\left(J -
q\right)^2 + \alpha_r\|\nabla q\|^2\right]. $$

Only first derivatives remain, so $\mathbf{u}$, $p$ and $q$ can all be interpolated with linear shape functions. The state of this formulation is therefore $\mathbf{m} = [\mathbf{u}, p, q]$, with the two auxiliary fields entering the regularization contribution $\Pi_r$. The displacement uses $Q_1$ elements on the full mesh, while $p$ and $q$ are $Q_1$ fields on the third medium submesh only, so that the additional unknowns stay confined to $\Omega_{tm}$. A Gauss-Lobatto rule of degree one is used here for the third medium elements.

The regularization paramaters are taken as $\beta_1 = 10^{4}$, $\beta_2 =10$ and $\alpha_r = 100$.

In [8]:
using Gridap.FESpaces

# -------------------------------------------------
# 6. Constitutive law – Neo-Hookean (per cell)
# -------------------------------------------------
#F(∇u) = one(∇u) + ∇u'
C(∇u) = F(∇u)' ⋅ F(∇u)
J(∇u) = sqrt(det(C(∇u)))

# Virtual Green-Lagrange strain increment
#δE(∇v, ∇u) = 0.5 * (∇v ⋅ F(∇u) + F(∇u)' ⋅ ∇v')
# -------------------------------------------------
# FE spaces
# -------------------------------------------------
# reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
# V = TestFESpace(model, reffe, conformity=:H1,
#                 dirichlet_tags=["Fixed", "Loads"])
# Element type
order = 1
degree = 2*order+2
dΩ = Measure(Ω, degree)

# FE space

reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags=["Fixed", "Loads"])
reffe_p = ReferenceFE(lagrangian, Float64, order)
V_p = TestFESpace(model, reffe_p, conformity=:L2)

# reffe_q = ReferenceFE(lagrangian, Float64, order)
# V_q = TestFESpace(model, reffe_q, conformity=:L2)

# Regularisation parameters
β₁ = 1.0e5             # (α - p)^2 coefficient
β₂ = 0.0             # (α - p)^2 coefficient
αᵣ = 100.0               # ∇p⋅∇p length sc

function tanϕ(∇u)
    F_val = F(∇u)
    a = F_val[1,2] - F_val[2,1]
    d = F_val[1,1] + F_val[2,2]
    return a / d
end    

# Assuming that the trace does not play a big role when it comes to severe element distortion, the denominator can be ingored in the numerical part.
# nominator in the term tanϕ is the skew (shear) part of the deformation gradient, while the denominator is its trace.

# G(∇u) = ∂tanϕ/∂(∇u)   ---> check this function
function G(∇u)
    F_val = F(∇u)
    n = F_val[1,2] - F_val[2,1]
    b = (F_val[1,1] + F_val[2,2])
    d = (F_val[1,1] + F_val[2,2] + 1e-12)^2    
    return TensorValue(-n/d, -b/d, b/d, -n/d )
end

function F_invt(∇u)
    return inv(F(∇u)')
end

#----------------------------------------------------
# Weak form (based on the principle of virtual work)
#----------------------------------------------------
function ℝ((u,p),(v,ξ))
    based_material = ∫((δE ∘ (∇(v), ∇(u))) ⊙ (S ∘ (∇(u), material_field)))dΩ 
    u_regularization = β₁*γ*∫((tanϕ ∘ ∇(u) - (1/d)*p)*(G∘∇(u) ⊙ ∇(v)))dΩ 
                    #+  β₂*γ*∫((J ∘ ∇(u) - q)*(J∘∇(u))*(F_invt∘∇(u) ⊙ ∇(v)))dΩ
    p_regularization = ∫( γ*(β₁*( (tanϕ ∘ ∇(u)) - (1/d)*p )*(ξ)*(-1/d) + αᵣ*∇(p)⋅∇(ξ)))dΩ 
    #q_regularization = ∫(γ*(β₂*( (J ∘ ∇(u)) - q )*(-η) + αᵣ*∇(q)⋅∇(η)))dΩ
    return based_material + u_regularization + p_regularization #+ q_regularization
end

function run_step_multi(x0, load_y, step, cache, fesolver, regularization_term)
    g_clamped = VectorValue(0.0, 0.0)
    g_loaded  = VectorValue(0.0, load_y)        # Vertical prescribed displacement

    U = TrialFESpace(V, [g_clamped, g_loaded])
    U_p = TrialFESpace(V_p)
    #U_q= TrialFESpace(V_q)

    # Multi-field spaces    
    X = MultiFieldFESpace([U, U_p])   # (u, p, q)
    Y = MultiFieldFESpace([V, V_p])   # (v, ξ, η)    

    op = FEOperator(ℝ, X, Y)
    xh = FEFunction(X, x0)
    # Safe solver execution tracking
    local solver_failed = false    

    try
        xh, cache = solve!(xh, solver, op, cache)
    catch
        solver_failed = true
    end
    uh, ph = xh
    r_vector = Gridap.FESpaces.residual(op, xh)
    r_norm = norm(r_vector)
    # Define your own success criteria
    # Usually, a residual < 1e-6 is excellent for non-linear mechanics
    if r_norm > 1e-6 || solver_failed
        println("!!! Step $step DIVERGED")
        return nothing, nothing, false
    end
    
    # ----- VTK output (includes material_id) -----
    data = ["u" => uh, "material_id"=> material_field]
    mkpath(regularization_term)
    writevtk(
        Ω,"$regularization_term/step_$(lpad(step,3,'0'))";
        cellfields = data
    )    
    return get_free_dof_values(xh), cache, true
end

function run_adaptive_loading_multi(fesolver, regularization_term, target_displacement, adaptive_load, initial_increment, min_increment, max_increment)
    max_load = target_displacement
    last_load = 0.0
    dload = initial_increment
    min_dload = min_increment
    max_dload = max_increment
    load = last_load + dload
    num_iteration = 0
    load_step = 1
    U_init = TrialFESpace(V)
    Up_init = TrialFESpace(V_p)
    #Uq_init = TrialFESpace(V_q)
    X_initial = MultiFieldFESpace([U_init, Up_init])
    x_converged = zero_free_values(X_initial)
    cache = nothing

    while abs(load) < abs(max_load)
        increment = sign(max_load)*min(dload, abs(max_load - load))
        trial_load = load + increment

        println("\n>>> LOAD STEP: u_y $trial_load (dload: $increment)")
        
        u_new, new_cache, success = run_step_multi(x_converged, trial_load, load_step, cache, fesolver,regularization_term)
        if success
            # Success: Move state forward
            load_step += 1
            x_converged = u_new
            cache = new_cache
            load = trial_load
            if abs(load) < abs(max_load)
                dload = (dload >= max_dload) ? dload : min(dload * 1.1, abs(max_load - load))
            end          
        else
            # Failure: Backtrack and cut load step
            println("!!! DIVERGENCE: Cutting load step size from $dload to $(dload/2)")
            dload /= 2
            if dload < min_dload
                error("Simulation aborted: Step size dropped below threshold.")
            end
        end    
    end
    return
end

#run_adaptive_loading_multi(solver, "Wrigger_regularization_out", -0.6, true, 0.004, 0.002, 0.005)

run_adaptive_loading_multi (generic function with 1 method)

### Triangular Element



In [9]:
# -------------------------------------------------
# Geometry – three rectangles (union = full domain)
# -------------------------------------------------
L, H = 1.20, 0.6                     # total size
h_bot = 0.2*H                       # yellow bottom thickness
h_mid = 0.6*H                       # brown middle thickness
h_top = 0.2*H                       # yellow top thickness
domain = (0, L, 0, H)

#3---6---4
#|       |
#7   9   8
#|       |
#1---5---2

γ  = 1.0e-6              # scaling parameter
β₁ = 1000000.0             # (α - p)^2 coefficient
αᵣ = 1000.0               # ∇p⋅∇p length scale

K₁ = 5.0/3.0;  μ₁ = 5/14; λ₁ = 30/21;

# Brown  (soft)    → K₂, μ₂
K₂ = K₁*γ;  μ₂ = μ₁*γ; λ₂ = λ₁*γ; 

K = 5.0/3.0;  μ = 5/14; λ = 30/21;

# -------------------------------------------------
# Discretisation
# -------------------------------------------------
partition = (30, 15)
d = L/30;
model_quad = CartesianDiscreteModel(domain, partition)
# Convert every quad → two triangles  ⇒ unstructured triangular mesh
model_tri = simplexify(model_quad)       # ← this is the key line

# -------------------------------------------------
# Tag cells by centroid Y-coordinate
# -------------------------------------------------
Ω = Triangulation(model_tri)
cell_points = get_cell_coordinates(Ω)          # 4 corner points per cell
cell_centroids = lazy_map(mean, cell_points)   # ← one centroid per cell

# Material IDs (1 = yellow, 2 = brown)
material_id = zeros(Int, num_cells(model_tri))
for (i, q) in enumerate(cell_centroids)
    (x,y) = q 
    if ((y ≤ h_bot || y ≥ (h_bot + h_mid) || x ≤ 0.2*H) && x ≤ 1.2)    # yellow layers
        material_id[i] = 1
    else                                                 # brown middle
        material_id[i] = 2
    end
end
material_field = CellField(material_id, Ω)

# Get all node coordinates
node_coords = Gridap.Geometry.get_node_coordinates(model_tri) 


# Find node index
let tip_node_id = nothing
for (i, p) in enumerate(node_coords)
    if norm(p - tip_point) < tol
        tip_node_id = i
        break
    end
end

@assert tip_node_id !== nothing "Tip node not found – check coordinates / tolerance"
    # ---- expose only the id to the outer scope ----
    global tip_node_id_global = tip_node_id
end

println("Prescribed displacement at node $tip_node_id_global: (1.2, 0.6)")

# -------------------------------------------------
# Boundary tags
# -------------------------------------------------
labels = get_face_labeling(model_tri)
entity = num_entities(labels)+1
labels.d_to_dface_to_entity[1][tip_node_id_global] = entity
add_tag!(labels,"Tip",[entity])
add_tag_from_tags!(labels, "clamped", [1, 3, 7])   # left edge + corners

# -------------------------------------------------
# Integration measures
# -------------------------------------------------
order = 1
degree = 2*order
dΩ = Measure(Ω, degree+2)

# -------------------------------------------------
# FE spaces
# -------------------------------------------------

reffe_u = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
V = TestFESpace(model_tri, reffe_u, conformity=:H1,
                dirichlet_tags=["clamped", "Tip"])

reffe_p = ReferenceFE(lagrangian, Float64, order)
V_p = TestFESpace(model_tri, reffe_p, conformity=:H1)


# Trial space will be built inside the incremental loop

reg_mask = CellField(material_id .== 2, Ω)

#run_adaptive_loading_multi(solver, "Wrigger_regularization_tri_out", -0.6, true, 0.004, 0.002, 0.005)

Prescribed displacement at node 496: (1.2, 0.6)


GenericCellField():
 num_cells: 900
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 12499978699498513324

### Three Dimensional Case - hexahedron and tetrahedron 

The TMC could be used for the three-dimensional case by using the skew part of the deformation gradient $\mathbf{F}^{skew} = \mathbf{F} - \mathbf{F}^{\top}$, which leads to the regularization
$$
    \Psi_{r}^{fskew} = \alpha_r \frac{\gamma}{2} \lVert \nabla_0 \left(\mathbf{F} - \mathbf{F}^{\top}\right) \lvert^2 = \alpha_r \frac{\gamma}{2} \lVert \nabla_0 \mathbf{F}^{skew} \lvert^2 
$$
The direct computation of the gradient of $\mathbf{F}^{skew}$ still involves the second derivative of $\mathbf{u}$. The approximation of these gradient is used by introduced the auxiliary variable $p_i$ with enhance-gradient technique. The regularization term can be split into the three components which define the skew symmetric part of the deformation gradient, $f_1^{skew} = F_{12} - F_{21}, f_2^{skew} = F_{13} - F_{31}, f_3^{skew} = F_{23} - F_{32}$. This leads to
$$
    \Psi_{r}^{\nabla F_i}(\mathbf{m}) = \gamma \left( \frac{\beta_r}{2} \left[ f_i^{skew} - \frac{1}{d} p_i \right]^2 + \frac{\alpha_r}{2} \lvert \nabla_0 p_i \lvert \right)
$$


In [10]:
# -------------------------------------------------
# 2. Geometry – three rectangles (union = full domain)
# -------------------------------------------------
L, H, T = 0.8, 0.28, 0.2                     # Length, Height, thickness (extrusion in Z)
h_bot = 0.2*T                                # yellow bottom thickness
h_mid = 0.6*H                                # brown middle thickness
h_top = 0.2*H                                # yellow top thickness
domain = (0, L, 0, H, 0, T)



# -------------------------------------------------
# 3. Discretisation
# -------------------------------------------------
partition = (20, 7, 5)                         # (nx, ny, nz)
model = CartesianDiscreteModel(domain, partition)

d = L/20;

# -------------------------------------------------
# 4. Tag cells by centroid Y-coordinate
# -------------------------------------------------
Ω = Triangulation(model)
cell_points = get_cell_coordinates(Ω)          # 4 corner points per cell
cell_centroids = lazy_map(mean, cell_points)   # ← one centroid per cell


# Material IDs (1 = yellow, 2 = brown)
material_id = zeros(Int, num_cells(model))
for (i, c) in enumerate(cell_centroids)
    (x, y, z) = c 
    if ((y ≤ h_bot || y ≥ (0.24) || x ≤ 0.2*T) && x ≤ 1.2)    # yellow layers
        material_id[i] = 1
    else                                                 # brown middle
        material_id[i] = 2
    end
end
material_field = CellField(material_id, Ω)   # for VTK
# -------------------------------------------------
#  Write VTK file — open immediately in Paraview!
# -------------------------------------------------
#writevtk(Ω, "C_shape_3D_geometry",
#    cellfields = ["material_id" => material_field])

# -------------------------------------------------
# 5. Material parameters (per cell)
# -------------------------------------------------
# Regularisation parameters
γ  = 1.0e-6                # scaling factor
β₁ = 1000000.0             # (α - p)^2 coefficient
αᵣ = 1000.0                # ∇p⋅∇p length 

# -------------------------------------------------
# 7. Boundary tags
# -------------------------------------------------
# Tag the left face (x = 0); right face (x = L)
labels = get_face_labeling(model)
add_tag_from_tags!(labels,"clamped", [1,3,5,7,13,15,17,19,25]); # left face: corners (1,3,5,7); edges (13,15,17,19); others (25) 
add_tag_from_tags!(labels, "load", [4,8,20]); # Right face: corners (2,4,6,8); edges (14,16,18,20); others (26) 

# -------------------------------------------------
# 6. Constitutive law – Neo-Hookean (per cell)
# -------------------------------------------------
F(∇u) = one(∇u) + ∇u'
C(∇u) = F(∇u)' ⋅ F(∇u)
J(∇u) = sqrt(det(C(∇u)))

# Virtual Green-Lagrange strain increment
δE(∇v, ∇u) = 0.5 * (∇v ⋅ F(∇u) + F(∇u)' ⋅ ∇v')
# -------------------------------------------------
# 8. Integration measures
# -------------------------------------------------
order = 1
degree = 2*order
dΩ = Measure(Ω, degree+2)

# -------------------------------------------------
# 9. FE spaces
# -------------------------------------------------
reffe_u = ReferenceFE(lagrangian, VectorValue{3,Float64}, order)
V = TestFESpace(model, reffe_u, conformity=:H1,
                dirichlet_tags=["clamped", "load"])

reffe_p1 = ReferenceFE(lagrangian, Float64, order)
V_p1 = TestFESpace(model, reffe_p1, conformity=:L2)

reffe_p2 = ReferenceFE(lagrangian, Float64, order)
V_p2 = TestFESpace(model, reffe_p2, conformity=:L2)

reffe_p3 = ReferenceFE(lagrangian, Float64, order)
V_p3 = TestFESpace(model, reffe_p3, conformity=:L2)

Y = MultiFieldFESpace([V, V_p1, V_p2, V_p3])   # (v, ξ₁, ξ₂, ξ₃)

# Trial space will be built inside the incremental loop

reg_mask = CellField(material_id .== 2, Ω)

# ==============================================================================
# In 3D, the regularization term split into the three components which
# define the skew symmetric part of deformation gradient
# ==============================================================================
function fskew₁(∇u)
    F_val = F(∇u)
    fs₁ =  F_val[1,2] - F_val[2,1]
    return fs₁
end 

function fskew₂(∇u)
    F_val = F(∇u)
    fs₂ =  F_val[1,3] - F_val[3,1]
    return fs₂
end 

function fskew₃(∇u)
    F_val = F(∇u)
    fs₃ =  F_val[2,3] - F_val[3,2]
    return fs₃
end 

const ∂fskew₁_∂F = TensorValue(0, 1, 0, -1, 0, 0, 0, 0, 0)

const ∂fskew₂_∂F = TensorValue(0, 0, 1, 0, 0, 0, -1, 0, 0)

const ∂fskew₃_∂F = TensorValue(0, 0, 0, 0, 0, 1, 0, -1, 0)


# -------------------------------------------------
# 10. Weak form – material parameters vary per cell
# -------------------------------------------------

function ℝ((u,p1,p2,p3),(v,ξ1,ξ2,ξ3))
    hyperelas = ∫((δE ∘ (∇(v), ∇(u))) ⊙ (S ∘ (∇(u), material_field)))dΩ
    #reg_u = ∫( Reg ∘ (∇(u),p)⊙∇(v)*reg_mask)dΩ
    #reg_u = ∫(( 5e-4     *(∇(u) ⊙ ∇(v) )) *reg_mask)dΩ
    #hyperelas = ∫((∇(v))⊙(P ∘ (∇(u), material_field)))dΩ 
    reg_u1 = β₁*γ*∫((fskew₁ ∘ ∇(u) - (1/d)*p1)*(∂fskew₁_∂F ⊙ ∇(v)))dΩ 
    reg_u2 = β₁*γ*∫((fskew₂ ∘ ∇(u) - (1/d)*p2)*(∂fskew₂_∂F ⊙ ∇(v)))dΩ
    reg_u3 = β₁*γ*∫((fskew₃ ∘ ∇(u) - (1/d)*p3)*(∂fskew₃_∂F ⊙ ∇(v)))dΩ  
    reg_p1 = ∫( γ*(β₁*( (fskew₁ ∘ ∇(u)) - (1/d)*p1 )*(-1/d)*ξ1 + αᵣ*∇(p1)⋅∇(ξ1)))dΩ
    reg_p2 = ∫( γ*(β₁*( (fskew₂ ∘ ∇(u)) - (1/d)*p2 )*(-1/d)*ξ2 + αᵣ*∇(p2)⋅∇(ξ2)))dΩ
    reg_p3 = ∫( γ*(β₁*( (fskew₃ ∘ ∇(u)) - (1/d)*p3 )*(-1/d)*ξ3 + αᵣ*∇(p3)⋅∇(ξ3)))dΩ
    return hyperelas + reg_p1 + reg_p2 + reg_p3 + reg_u1 + reg_u2 + reg_u3
end

function run_step_3D(x0, load_y, step, cache, fesolver, regularization_term)
    g_clamped = VectorValue(0.0, 0.0, 0.0)
    g_loaded  = VectorValue(0.0, load_y, 0.0)        # Vertical prescribed displacement

    U = TrialFESpace(V, [g_clamped, g_loaded])
    U_p1 = TrialFESpace(V_p1)
    U_p2 = TrialFESpace(V_p2)
    U_p3 = TrialFESpace(V_p3)

    
    # Multi-field spaces    
    X = MultiFieldFESpace([U, U_p1, U_p2, U_p3])   # (u, p₁, p₂, p₃)
      

    op = FEOperator(ℝ, X, Y)
    xh = FEFunction(X, x0)
    # Safe solver execution tracking
    local solver_failed = false    
    
    try
        xh, cache = solve!(xh, solver, op, cache)
    catch
        solver_failed = true
    end
    uh, ph1, ph2, ph3 = xh
    r_vector = Gridap.FESpaces.residual(op, xh)
    r_norm = norm(r_vector)
    # Define your own success criteria
    # Usually, a residual < 1e-6 is excellent for non-linear mechanics
    if r_norm > 1e-6 || solver_failed
        println("!!! Step $step DIVERGED")
        return nothing, nothing, false
    end
    
    # ----- VTK output (includes material_id) -----
    data = ["u" => uh, "material_id"=> material_field, "p1" => ph1, "p2" => ph2, "p3" => ph3]
    mkpath(regularization_term)
    writevtk(
        Ω,"$regularization_term/step_$(lpad(step,3,'0'))";
        cellfields = data
    )    
    return get_free_dof_values(xh), cache, true
end

function run_adaptive_loading_3D(fesolver, regularization_term, target_displacement, adaptive_load, initial_increment, min_increment, max_increment)
    max_load = target_displacement
    last_load = 0.0
    dload = initial_increment
    min_dload = min_increment
    max_dload = max_increment
    load = last_load + dload
    num_iteration = 0
    load_step = 1
    U_init = TrialFESpace(V)
    Up1_init = TrialFESpace(V_p1)
    Up2_init = TrialFESpace(V_p2)
    Up3_init = TrialFESpace(V_p3)
    X_initial = MultiFieldFESpace([U_init, Up1_init, Up2_init, Up3_init])
    x_converged = zero_free_values(X_initial)
    cache = nothing

    while abs(load) < abs(max_load)
        increment = sign(max_load)*min(dload, abs(max_load - load))
        trial_load = load + increment

        println("\n>>> LOAD STEP: u_y $trial_load (dload: $increment)")
        
        u_new, new_cache, success = run_step_3D(x_converged, trial_load, load_step, cache, fesolver,regularization_term)
        if success
            # Success: Move state forward
            load_step += 1
            x_converged = u_new
            cache = new_cache
            load = trial_load
            if abs(load) < abs(max_load)
                dload = (dload >= max_dload) ? dload : min(dload * 1.1, abs(max_load - load))
            end          
        else
            # Failure: Backtrack and cut load step
            println("!!! DIVERGENCE: Cutting load step size from $dload to $(dload/2)")
            dload /= 2
            if dload < min_dload
                error("Simulation aborted: Step size dropped below threshold.")
            end
        end    
    end
    return
end

#run_adaptive_loading_3D(solver, "Wrigger_regularization_hex_out", -0.3, true, 0.004, 0.002, 0.005)

run_adaptive_loading_3D (generic function with 1 method)

In [11]:
# -------------------------------------------------
# Discretisation – 3D hex mesh → 3D tet mesh
# -------------------------------------------------

model_hex = CartesianDiscreteModel(domain, partition)
model_tet = simplexify(model_hex)

d = L/40;

# -------------------------------------------------
# Tag cells by centroid Y-coordinate
# -------------------------------------------------
Ω = Triangulation(model_tet)
cell_points = get_cell_coordinates(Ω)          # 4 corner points per cell
cell_centroids = lazy_map(mean, cell_points)   # ← one centroid per cell


# Material IDs (1 = yellow, 2 = brown)
material_id = zeros(Int, num_cells(model_tet))
for (i, c) in enumerate(cell_centroids)
    (x, y, z) = c 
    if ((y ≤ h_bot || y ≥ (0.24) || x ≤ 0.2*T) && x ≤ 1.2)    # yellow layers
        material_id[i] = 1
    else                                                 # brown middle
        material_id[i] = 2
    end
end
material_field = CellField(material_id, Ω)   # for VTK

# Tag the left face (x = 0); right face (x = L)
labels = get_face_labeling(model_tet)
add_tag_from_tags!(labels,"clamped", [1,3,5,7,13,15,17,19,25]); # left face: corners (1,3,5,7); edges (13,15,17,19); others (25) 
add_tag_from_tags!(labels, "load", [4,8,20]); # Right face: corners (2,4,6,8); edges (14,16,18,20); others (26) 

# -------------------------------------------------
# Integration measures
# -------------------------------------------------
order = 1
degree = 2*order
dΩ = Measure(Ω, degree+2)

# -------------------------------------------------
# FE spaces
# -------------------------------------------------
reffe_u = ReferenceFE(lagrangian, VectorValue{3,Float64}, order)
V = TestFESpace(model_tet, reffe_u, conformity=:H1,
                dirichlet_tags=["clamped", "load"])

reffe_p1 = ReferenceFE(lagrangian, Float64, order)
V_p1 = TestFESpace(model_tet, reffe_p1, conformity=:H1)

reffe_p2 = ReferenceFE(lagrangian, Float64, order)
V_p2 = TestFESpace(model_tet, reffe_p2, conformity=:H1)

reffe_p3 = ReferenceFE(lagrangian, Float64, order)
V_p3 = TestFESpace(model_tet, reffe_p3, conformity=:H1)

Y = MultiFieldFESpace([V, V_p1, V_p2, V_p3])   # (v, ξ₁, ξ₂, ξ₃)

# Trial space will be built inside the incremental loop

reg_mask = CellField(material_id .== 2, Ω)

#run_adaptive_loading_3D(solver, "Wrigger_regularization_tet_out", -0.3, true, 0.004, 0.002, 0.005)

GenericCellField():
 num_cells: 4200
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 5817734167929826621

## Deformation gradient-based averaging regularization

A new regularization method has recently been introduced by [[4](
https://doi.org/10.48550/arXiv.2606.28036)] completely avoids the use of second gradient of displacement. Instead, building upon the Hu-Hu regularization, then an auxiliary-field stabilization is introduced in which a deformation-gradient-like field $\boldsymbol{\Theta}$ is treated as an independent unknown in the third medium and coupled to the physical deformation gradient by a penalty term.
$$ \tilde{\Psi}_r(\boldsymbol{u}, \boldsymbol{\Theta}) =
\frac{p_{\Theta}}{2}\left\|\boldsymbol{\Theta} - \boldsymbol{F}\right\|^{2}, $$

where $p_{\Theta}$ is penalty parameter. The regularization is then applied directly to the gradient of $\boldsymbol{\Theta}$ yielding the complete regularization contribution

$$ \Psi_r(\boldsymbol{u}, \boldsymbol{\Theta}) = \frac{p_{\Theta}}{2}\left\|\boldsymbol{\Theta} - \boldsymbol{F}\right\|^{2} + \frac{\alpha_r}{2}\left\|\nabla\boldsymbol{\Theta}\right\|^{2}, $$

with $\alpha_r$ denoting the regularization parameter. Because only first derivative of $\boldsymbol{\Theta}$ appear, a gradient control of $\mathbf{F}$ is covered without requiring second derivatives of the displacement field $\mathbf{u}$. Consequently, standard first-order element become admissible for both fields. The resulting state vector is $\mathbf{m} = [\mathbf{m}, \boldsymbol{\Theta} ] $. The displacement is discretized with continuous $Q_1$ elements over the entire mesh, while $\boldsymbol{\Theta}$ is represented as $Q_1$ tensor-valued field restricted to the third-medium submesh.

In the present work, the regularization parameters are set to $p_{\Theta} = 5 \, 10^{-2}$ and $\alpha_r = 10^{-8}$.

In [15]:
# Dimension of the model
D = 2

#-----------------------------------------------------------------------
# Geometry - C_shape body
#-----------------------------------------------------------------------
L, H = 1.24, 0.6                    #length, and height
h_bot = 0.2*H                       # yellow bottom thickness
h_mid = 0.6*H                       # brown middle thickness
h_top = 0.2*H                       # yellow top thickness
d = L/30;

# Yellow (stiff)   → K₁, μ₁
γ  = 1.0e-5              # scaling parameter
K₁ = 5.0/3.0;  μ₁ = 5/14; λ₁ = 30/21;

# Brown  (soft)    → K₂, μ₂
K₂ = K₁*γ;  μ₂ = μ₁*γ; λ₂ = λ₁*γ; 

K = 5.0/3.0;  μ = 5/14; λ = 30/21;
# Regularisation parameters

#-----------------------------------------------------------------------
# Discretisation
#-----------------------------------------------------------------------
Nx, Ny = 31, 15                     # cells along x and y within C-shape
domain = (0, L, 0, H)
partition = (Nx,Ny)
model = CartesianDiscreteModel(domain, partition)
Ω = Triangulation(model)

# Tag cells by centroid coordinate - mark cells
cell_points = get_cell_coordinates(Ω)          # 4 corner points per cell
cell_centroids = lazy_map(mean, cell_points)   # ← one centroid per cell
centroid_y = get_array(lazy_map(p -> p[2], cell_centroids))
# Material IDs (1 = solid, 2 = third medium)
material_id = zeros(Int, num_cells(model))
for (i, q) in enumerate(cell_centroids)
    (x,y) = q 
    if ((y ≤ h_bot || y ≥ (h_bot + h_mid) || x ≤ 0.2*H) && x ≤ 1.2)    # yellow layers
        material_id[i] = 1
    else                                                 # brown middle
        material_id[i] = 2
    end
end
material_field = CellField(material_id, Ω);   # for VTK

#-----------------------------------------------------------------------
# Find node by coordinate (tolerance = 10e-6)
#-----------------------------------------------------------------------

tol = 1e-6
tip_point = Point(1.2, 0.6)                 # (x,y) coordinate of node

# Get all node coordinates
node_coords = Gridap.Geometry.get_node_coordinates(model) 

# Find node index
let tip_node_id = nothing
for (i, p) in enumerate(node_coords)
    if norm(p - tip_point) < tol
        tip_node_id = i
        break
    end
end

@assert tip_node_id !== nothing "Tip node not found – check coordinates / tolerance"
    # ---- expose only the id to the outer scope ----
    global tip_node_id_global = tip_node_id
end

println("Prescribed displacement at node $tip_node_id_global: (1.2, 0.5)")


# -------------------------------------------------
# 6. Constitutive law – Neo-Hookean (per cell)
# -------------------------------------------------
F(∇u) = one(∇u) + ∇u'
C(∇u) = F(∇u)' ⋅ F(∇u)
J(∇u) = sqrt(det(C(∇u)))

# Virtual Green-Lagrange strain increment
δE(∇v, ∇u) = 0.5 * (∇v ⋅ F(∇u) + F(∇u)' ⋅ ∇v')

# Second Piola-Kirchhoff stress
function S(∇u, tag)
    λ, μ = (tag == 1) ? (λ₁, μ₁) : (λ₂, μ₂)
    Cinv = inv(C(∇u))
    return μ*(one(∇u) - Cinv) + λ*log(J(∇u))*Cinv
end

# Element type
order = 1
degree = 2*order+2
dΩ = Measure(Ω, degree)

#-----------------------------------------------------------------------
# Boundary condition tags
#-----------------------------------------------------------------------

labels = get_face_labeling(model)
entity = num_entities(labels)+1
labels.d_to_dface_to_entity[1][tip_node_id_global] = entity
add_tag!(labels,"Loads",[entity])
add_tag_from_tags!(labels, "Fixed", [1, 3, 7])   # left edge + corners

# Displacemence and deformation gradient averaging at the centroid of the element
reffe_u = ReferenceFE(lagrangian, VectorValue{D,Float64}, order)
V = TestFESpace(model, reffe_u, conformity=:H1,
                dirichlet_tags=["Fixed", "Loads"])
reffe_θ = ReferenceFE(lagrangian, TensorValue{D,D,Float64},order)
V_θ = TestFESpace(model, reffe_θ, conformity=:H1)

#----------------------------------------------------
# Weak form (based on the principle of virtual work)
#----------------------------------------------------
function ℝ((u,θ),(v,w))
    ∇u = ∇(u)
    ∇v = ∇(v)

    # Compose F and δF pointwise
    F_val = F∘(∇u)
    κ = 5.0e-3 # Pa
    α_r = 1.0e-9
    based_material = ∫((δE ∘ (∇(v), ∇(u))) ⊙ (S ∘ (∇(u), material_field)))dΩ
    
    # deformation gradient averaging at the centroid of the element
    # Ψᵣ = 0.5 * κ * ||F - Θ||² 
    # The first variation of the regularization term
    δW_r = ∫( κ * (F_val - θ) ⊙ (∇v) + κ * (θ - F_val) ⊙ w )dΩ
    δW_regularization = ∫(α_r*(∇(θ) ⊙ ∇(w)))dΩ

    return based_material + δW_r + δW_regularization
end

#----------------------------------------------------
# Solver - Newton solver with line search
#----------------------------------------------------

nls = NLSolver(show_trace=true, method=:newton,
    linesearch= BackTracking(), iterations = 200)

solver = FESolver(nls)

# Setup adaptive loading strategy or constant loading step

function run_step_Θ(x0, load_y, step, cache, fesolver, regularization_term)
    g_clamped = VectorValue(0.0, 0.0)
    g_loaded  = VectorValue(0.0, load_y)        # Vertical prescribed displacement

    U = TrialFESpace(V, [g_clamped, g_loaded])
    Θ = TrialFESpace(V_θ)

    # Multi-field spaces    
    X = MultiFieldFESpace([U, Θ])   # (u, θ)
    Y = MultiFieldFESpace([V, V_θ])   # (v, w)    

    op = FEOperator(ℝ, X, Y)
    xh = FEFunction(X, x0)
    # Safe solver execution tracking
    local solver_failed = false    

    try
        xh, cache = solve!(xh, solver, op, cache)
    catch
        solver_failed = true
    end
    uh, θh = xh
    r_vector = Gridap.FESpaces.residual(op, xh)
    r_norm = norm(r_vector)
    # Define your own success criteria
    # Usually, a residual < 1e-6 is excellent for non-linear mechanics
    if r_norm > 1e-6 || solver_failed
        println("!!! Step $step DIVERGED")
        return nothing, nothing, false
    end
    
    # ----- VTK output (includes material_id) -----
    data = ["u" => uh, "material_id"=> material_field]
    mkpath(regularization_term)
    writevtk(
        Ω,"$regularization_term/step_$(lpad(step,3,'0'))";
        cellfields = data
    )    
    return get_free_dof_values(xh), cache, true
end

function run_adaptive_loading_Θ(fesolver, regularization_term, target_displacement, adaptive_load, initial_increment, min_increment, max_increment)
    max_load = target_displacement
    last_load = 0.0
    dload = initial_increment
    min_dload = min_increment
    max_dload = max_increment
    load = last_load + dload
    num_iteration = 0
    load_step = 1
    U_init = TrialFESpace(V, [VectorValue(0.0, 0.0), VectorValue(0.0, 0.0)])
    Θ_init = TrialFESpace(V_θ)
    X_initial = MultiFieldFESpace([U_init, Θ_init])
    #Uq_init = TrialFESpace(V_q)
    I_tensor = one(TensorValue{D, D, Float64})
    u_init = zero(U_init)
    θ_init = interpolate(I_tensor, Θ_init)
    u_0 = VectorValue(0.0, 0.0)
    xh_init = interpolate([u_0, I_tensor], X_initial)
    x_converged = get_free_dof_values(xh_init)
    cache = nothing

    while abs(load) < abs(max_load)
        increment = sign(max_load)*min(dload, abs(max_load - load))
        trial_load = load + increment

        println("\n>>> LOAD STEP: u_y $trial_load (dload: $increment)")
        
        u_new, new_cache, success = run_step_Θ(x_converged, trial_load, load_step, cache, fesolver,regularization_term)
        if success
            # Success: Move state forward
            load_step += 1
            x_converged = u_new
            cache = new_cache
            load = trial_load
            if abs(load) < abs(max_load)
                dload = (dload >= max_dload) ? dload : min(dload * 1.1, abs(max_load - load))
            end          
        else
            # Failure: Backtrack and cut load step
            println("!!! DIVERGENCE: Cutting load step size from $dload to $(dload/2)")
            dload /= 2
            if dload < min_dload
                error("Simulation aborted: Step size dropped below threshold.")
            end
        end    
    end
    return
end
run_adaptive_loading_Θ(solver, "gradient_based_regularization_out", -0.6, true, 0.004, 0.002, 0.005)

Prescribed displacement at node 511: (1.2, 0.5)

>>> LOAD STEP: u_y 0.0 (dload: -0.004)
Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     0.000000e+00              NaN

>>> LOAD STEP: u_y -0.0044 (dload: -0.0044)
Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     3.233465e-03              NaN
     1     3.149574e-04     5.871933e-02
     2     1.520857e-05     1.716324e-02
     3     2.137304e-09     4.299518e-06

>>> LOAD STEP: u_y -0.009240000000000002 (dload: -0.0048400000000000006)
Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     3.595774e-03              NaN
     1     3.797811e-04     8.585261e-02
     2     2.730249e-05     3.066928e-02
     3     9.150453e-09     2.158914e-05

>>> LOAD STEP: u_y -0.014564000000000002 (dload: -0.005324000000000001)
Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     4.003298e-03        

Using exactly $\mathbf{F} = \mathbf{I} + \nabla_0 \mathbf{u}$

## Reference

[1] G. Bluhm, O. Sigmund, and K. Poulios. **Internel contact modeling for finite strain topology optimization**. *Computational Mechanics*. https://doi.org/10.48550/arXiv.2010.14277 (2021)

[2] A. H. Frederiksen, A. Dalklint, O. Sigmund, and K. Poulios. **Improved third medium formulation for 3D topology optimization with contact**. *Computational Methods in Applied Mechanics and Engineering*. https://doi.org/https://doi.org/10.1016/j.cma.2024.117595 (2025)

[3] P. Wrigger, J. Korelc, and P. Junker. **First order finite element formulations for third medium contact**. *Computational Mechanics*. https://doi.org/10.1007/s00466-025-02628-y (2025)

[4] M. Vorwerk, J. Schoroder, P. Wriiter. **A robust mixed finite element formulation for third medium contact**. *Numerical Analysis*. https://doi.org/10.48550/arXiv.2606.28036